[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vladimiracunadev-create/python-data-science-bootcamp/blob/master/classes/11-evaluacion-y-pipelines/soluciones.ipynb)

# SOLUCIONES — Clase 11: Evaluacion Robusta y Pipelines de ML

> **Instrucciones:** Este archivo contiene las soluciones completas. Intenta resolver el `notebook.ipynb` antes de consultar estas soluciones.

---

## Objetivos

- Evaluar modelos con Cross-Validation robusta
- Construir Pipelines seguros contra data leakage
- Optimizar hiperparametros con GridSearchCV
- Diagnosticar overfitting y underfitting


## El problema con un solo split

```
PROBLEMA CON UN SOLO SPLIT
===========================

Split A (seed=1): Test accuracy = 0.81
Split B (seed=2): Test accuracy = 0.74
Split C (seed=3): Test accuracy = 0.87

Cual es el verdadero performance? NINGUNO por si solo.

SOLUCION: Cross-Validation (evaluar K veces)
```


In [ ]:
# ============================================================
# SOLUCION: Importaciones
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import (
    train_test_split,
    cross_val_score,
    StratifiedKFold,
    GridSearchCV,
    learning_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score

print("Librerias cargadas correctamente")


In [ ]:
# ============================================================
# SOLUCION: Carga de datos y creacion de X, y
# ============================================================

# Cargar dataset
df = pd.read_csv("../../datasets/retencion_clientes.csv")  # CSV de retencion
print(f"Dataset: {df.shape}")

# Seleccionar features numericas sin el target
cols_num = df.select_dtypes(include=["number"]).columns.tolist()  # solo numericas
features = [c for c in cols_num if c != "churned"]  # excluir target

X = df[features]    # matrix de features
y = df["churned"]   # vector target

print(f"Features: {features}")
print(f"Shape X: {X.shape}, Shape y: {y.shape}")

# Separar holdout ANTES de cualquier otra cosa
X_cv, X_holdout, y_cv, y_holdout = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y  # 20% holdout intocable
)
print(f"\nCV set: {X_cv.shape[0]} filas | Holdout: {X_holdout.shape[0]} filas")


## K-Fold Cross Validation

```
K-FOLD CROSS VALIDATION (K=5)
==============================

Iteracion 1: [ TEST  ][ TRAIN ][ TRAIN ][ TRAIN ][ TRAIN ]  score_1 = 0.82
Iteracion 2: [ TRAIN ][ TEST  ][ TRAIN ][ TRAIN ][ TRAIN ]  score_2 = 0.79
Iteracion 3: [ TRAIN ][ TRAIN ][ TEST  ][ TRAIN ][ TRAIN ]  score_3 = 0.84
Iteracion 4: [ TRAIN ][ TRAIN ][ TRAIN ][ TEST  ][ TRAIN ]  score_4 = 0.81
Iteracion 5: [ TRAIN ][ TRAIN ][ TRAIN ][ TRAIN ][ TEST  ]  score_5 = 0.80

Score final = mean(0.82, 0.79, 0.84, 0.81, 0.80) = 0.812 +/- 0.018
```


In [ ]:
# ============================================================
# SOLUCION: Cross-Validation con StratifiedKFold
# ============================================================

# Modelo base para evaluar
modelo_base = DecisionTreeClassifier(max_depth=5, random_state=42)

# StratifiedKFold: K divisiones preservando balance de clases en cada fold
kfold = StratifiedKFold(
    n_splits=5,      # 5 iteraciones
    shuffle=True,    # mezclar antes de dividir
    random_state=42  # reproducibilidad de la mezcla
)

# cross_val_score: entrena y evalua K veces automaticamente
# scoring='f1' usa el F1 de la clase positiva (churners)
cv_scores = cross_val_score(
    modelo_base,   # modelo a evaluar
    X_cv,          # features del conjunto de CV
    y_cv,          # target del conjunto de CV
    cv=kfold,      # estrategia de division
    scoring="f1"   # metrica: F1 de clase positiva
)

# Resultados
print("Scores F1 por fold:")
for i, s in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {s:.4f}")

print(f"\nMedia:   {cv_scores.mean():.4f}")
print(f"Std:     {cv_scores.std():.4f}")
print(f"Intervalo: [{cv_scores.mean()-cv_scores.std():.4f}, {cv_scores.mean()+cv_scores.std():.4f}]")


In [ ]:
# ============================================================
# SOLUCION: Grafico de barras de scores por fold
# ============================================================

# Etiquetas y colores diferenciados por rendimiento
folds = [f"Fold {i}" for i in range(1, len(cv_scores) + 1)]
colores = ["steelblue" if s >= cv_scores.mean() else "salmon" for s in cv_scores]

fig, ax = plt.subplots(figsize=(8, 5))

# Barras de scores
barras = ax.bar(folds, cv_scores, color=colores, edgecolor="white", width=0.6)

# Linea de la media
ax.axhline(y=cv_scores.mean(), color="darkblue", linewidth=2.5, linestyle="--",
           label=f"Media = {cv_scores.mean():.4f}")

# Etiquetas encima de cada barra
for barra, score in zip(barras, cv_scores):
    ax.text(barra.get_x() + barra.get_width()/2, score + 0.003,
            f"{score:.3f}", ha="center", fontsize=10, fontweight="bold")

ax.set_xlabel("Fold", fontsize=12)
ax.set_ylabel("F1 Score", fontsize=12)
ax.set_title("Cross-Validation: F1 Score por Fold", fontsize=13, fontweight="bold")
ax.set_ylim(0, 1.1)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()


## Diagnostico de Overfitting

```
DIAGNOSTICO DE OVERFITTING
===========================

Train Score  |  Test Score  |  Brecha   |  Diagnostico
-------------|--------------|-----------|------------------------
    0.90     |    0.88      | pequena   |  OK  Generaliza bien
    0.95     |    0.85      | media     |  AVISO  Revisar modelo
    0.99     |    0.65      | grande    |  ERROR  Overfitting
    0.65     |    0.63      | muy peq.  |  ERROR  Underfitting
```


In [ ]:
# ============================================================
# SOLUCION: Curva overfitting por max_depth
# ============================================================

# Rango de profundidades a explorar
profundidades = list(range(1, 16))  # de 1 a 15
train_scores  = []  # F1 en train
test_scores   = []  # F1 en test

# Split simple para comparacion train vs test
X_tr, X_te, y_tr, y_te = train_test_split(
    X_cv, y_cv, test_size=0.2, random_state=42, stratify=y_cv
)

# Entrenar y evaluar para cada profundidad
for depth in profundidades:
    dt = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt.fit(X_tr, y_tr)  # entrenar
    train_scores.append(f1_score(y_tr, dt.predict(X_tr), average="weighted"))  # F1 train
    test_scores.append(f1_score(y_te, dt.predict(X_te), average="weighted"))   # F1 test

# Graficar las dos curvas
fig, ax = plt.subplots(figsize=(10, 6))

ax.plot(profundidades, train_scores, "o-", color="steelblue",
        label="Train F1", linewidth=2, markersize=6)  # curva train
ax.plot(profundidades, test_scores, "s--", color="darkorange",
        label="Test F1", linewidth=2, markersize=6)    # curva test
ax.fill_between(profundidades, train_scores, test_scores,
                alpha=0.1, color="red", label="Zona overfitting")  # zona critica

ax.set_xlabel("Profundidad (max_depth)", fontsize=12)
ax.set_ylabel("F1 Score (weighted)", fontsize=12)
ax.set_title("Overfitting: Train vs Test F1 por max_depth", fontsize=13, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(profundidades)

plt.tight_layout()
plt.show()

# Identificar optimo
optimo = profundidades[test_scores.index(max(test_scores))]
print(f"Mejor max_depth: {optimo} | Train F1: {train_scores[optimo-1]:.4f} | Test F1: {max(test_scores):.4f}")


## Pipeline de ML

```
PIPELINE DE ML
==============

Sin Pipeline: riesgo de data leakage si se llama .fit() en datos de test.

Con Pipeline:
  pipe.fit(X_train) -> scaler.fit_transform(X_train) + modelo.fit()
  pipe.predict(X_test) -> scaler.transform(X_test) + modelo.predict()
  El scaler NUNCA hace fit con datos de test.
```


In [ ]:
# ============================================================
# SOLUCION: Pipeline StandardScaler + LogisticRegression
# ============================================================

# Construir el pipeline con dos pasos nombrados
pipeline = Pipeline([
    ("escalador", StandardScaler()),          # normalizar features
    ("modelo",    LogisticRegression(         # clasificador lineal
        max_iter=1000, random_state=42
    ))
])

# Dividir para demostrar el entrenamiento individual
X_tr, X_val, y_tr, y_val = train_test_split(
    X_cv, y_cv, test_size=0.2, random_state=42, stratify=y_cv
)

# Entrenar el pipeline completo
pipeline.fit(X_tr, y_tr)  # scaler.fit_transform(X_tr) + modelo.fit(X_scaled, y_tr)
print("Pipeline entrenado correctamente")

# Predecir sobre validacion
y_pred_pipe = pipeline.predict(X_val)  # scaler.transform(X_val) + modelo.predict()

# Reporte completo
print("\nClassification Report — Pipeline (StandardScaler + LogisticRegression):")
print("=" * 65)
print(classification_report(y_val, y_pred_pipe, target_names=["NO churn", "SI churn"]))


In [ ]:
# ============================================================
# SOLUCION: Pipeline dentro de cross_val_score
# ============================================================

# Combinar Pipeline + cross_val_score = practica correcta
# En cada fold, el scaler solo ve los datos de train del fold (sin leakage)
cv_scores_pipe = cross_val_score(
    pipeline,    # pipeline completo
    X_cv,        # features sin escalar (el pipeline lo hace)
    y_cv,        # target
    cv=kfold,    # 5-fold estratificado
    scoring="f1" # metrica F1
)

print("Pipeline CV Results:")
for i, s in enumerate(cv_scores_pipe, 1):
    print(f"  Fold {i}: {s:.4f}")

print(f"\n  Media:  {cv_scores_pipe.mean():.4f}")
print(f"  Std:    {cv_scores_pipe.std():.4f}")

print("\nComparacion:")
print(f"  Decision Tree: {cv_scores.mean():.4f} +/- {cv_scores.std():.4f}")
print(f"  LogReg Pipeline: {cv_scores_pipe.mean():.4f} +/- {cv_scores_pipe.std():.4f}")


## GridSearchCV

```
GRID SEARCH CROSS VALIDATION
=============================

param_grid = {'max_depth': [3, 5, 7], 'min_samples_leaf': [2, 5, 10]}

Combinaciones: 3 x 3 = 9
Con 5-fold CV: 9 x 5 = 45 entrenamientos

Resultado: la combinacion con el mejor score promedio de CV
```


In [ ]:
# ============================================================
# SOLUCION: GridSearchCV sobre DecisionTreeClassifier
# ============================================================

# Espacio de busqueda: cada clave = hiperparametro, cada valor = lista de opciones
param_grid = {
    "max_depth":         [3, 5, 7, 10],  # profundidades a comparar
    "min_samples_leaf":  [2, 5, 10],     # tamanos de hoja
    "min_samples_split": [2, 5, 10],     # tamanos de division de nodo
}

# Modelo base sin hiperparametros fijos
dt_base = DecisionTreeClassifier(random_state=42)

# GridSearchCV: probar todas las combinaciones con CV
grid_search = GridSearchCV(
    estimator=dt_base,   # modelo a optimizar
    param_grid=param_grid, # combinaciones a explorar
    cv=kfold,            # 5-fold estratificado
    scoring="f1",        # optimizar F1 de clase positiva
    n_jobs=-1,           # paralelizar en todos los nucleos
    verbose=1,           # mostrar progreso
    refit=True           # re-entrenar el mejor modelo con todos los datos de CV
)

print("Ejecutando GridSearchCV...")
grid_search.fit(X_cv, y_cv)  # probar 4x3x3=36 combinaciones x 5 folds = 180 modelos
print("Busqueda completada!")


In [ ]:
# ============================================================
# SOLUCION: Resultados de GridSearchCV
# ============================================================

# Mejores hiperparametros encontrados
print("Mejores hiperparametros:")
for param, valor in grid_search.best_params_.items():
    print(f"  {param}: {valor}")

print(f"\nMejor F1 en CV: {grid_search.best_score_:.4f}")

# El mejor modelo ya esta entrenado con todos los datos de CV
mejor_dt = grid_search.best_estimator_

# Top 5 combinaciones
df_grid = pd.DataFrame(grid_search.cv_results_)  # tabla completa de resultados
top5 = df_grid.sort_values("rank_test_score").head(5)
cols = ["params", "mean_test_score", "std_test_score", "rank_test_score"]
print("\nTop 5 combinaciones:")
print(top5[cols].to_string(index=False))


In [ ]:
# ============================================================
# SOLUCION: Evaluacion final en holdout
# ============================================================

# Usar el holdout por primera y unica vez
y_pred_holdout = mejor_dt.predict(X_holdout)  # predicciones sobre datos nunca vistos

# Comparar CV score estimado vs score real en holdout
f1_holdout = f1_score(y_holdout, y_pred_holdout, average="weighted")

print("Evaluacion final:")
print("=" * 50)
print(f"  F1 CV (estimado):         {grid_search.best_score_:.4f}")
print(f"  F1 Holdout (real):        {f1_holdout:.4f}")
diferencia = abs(grid_search.best_score_ - f1_holdout)
print(f"  Diferencia:               {diferencia:.4f}")

if diferencia < 0.05:
    print("  -> Estimacion CV precisa. Modelo confiable.")
else:
    print("  -> Diferencia notable. Revisar el proceso de evaluacion.")

print("\nClassification Report en holdout:")
print(classification_report(y_holdout, y_pred_holdout, target_names=["NO churn", "SI churn"]))


## Resumen de la Clase 11 — Checklist completado

- [x] Problema con un solo split (variabilidad por seed)
- [x] `StratifiedKFold` con 5 folds y shuffle
- [x] `cross_val_score` con scoring='f1'
- [x] Visualizacion de scores por fold (barras con colores)
- [x] Diagnostico overfitting: curva train vs test por max_depth
- [x] Pipeline: StandardScaler + LogisticRegression
- [x] Pipeline dentro de cross_val_score (sin data leakage)
- [x] `GridSearchCV` con 4x3x3=36 combinaciones
- [x] Evaluacion final honesta en holdout set

**Proxima clase:** Proyecto Final — integracion completa en analisis de ventas.
